# Lesson 9, one call at a time

Lesson 9 went from one unit, which is one straight line, to a hidden layer that
gives the data new coordinates; then it trained networks, by hand and with
scikit-learn and Keras, on Meridian's sensors and on the handwritten digits, and
watched training go wrong in four ways. This notebook takes each step on its own,
on a point or two where a pencil can follow it and on the lesson's own data where
the slides' numbers come from, so that the slide's number and the call that makes
it sit side by side.

**How to use it.** Each section is one *atom*. Before each code cell that teaches
something there is a question under *Your prediction, before you run the next
cell*: answer it first, then run. Notebook `00_the_scikit_learn_api.ipynb`
explains `fit`, `predict`, `predict_proba` and `random_state`; this one assumes
them.

Every atom names the slide it comes from, by title, and lesson 9's three notebooks
treat each idea at full scale. Atoms 1-7 can replace notebook 1 in class, atoms
8-12 notebook 2, atoms 13-17 notebook 3. The whole notebook runs in about a minute.

## Setup

In [1]:
# Everything used here ships with the course image:
# pip install numpy==1.26.4 pandas==2.1.1 scikit-learn==1.3.1 tensorflow-cpu==2.21.0

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # keep TensorFlow's C++ log quiet
import warnings

import numpy as np
import tensorflow as tf
import keras
from keras import layers

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Lesson 9's data: Meridian's acceptance test (3,000 sensors, two offsets),
# the channel-drift problem (800 sensors) and the 8x8 handwritten digits.
from instrument_data import load_digit_images, make_acceptance_test, make_channel_drift

# MLPClassifier warns when it stops at max_iter; the runs below are long enough.
warnings.filterwarnings("ignore", message="Stochastic Optimizer")
# The seed fixes starting weights and shuffling; determinism fixes the order of sums.
tf.config.experimental.enable_op_determinism()
np.set_printoptions(precision=4, suppress=True)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def relu(z):
    return np.maximum(0, z)


# The acceptance test, split and standardised exactly as notebook 01 does it.
units = make_acceptance_test()
X = units[["gain_offset_db", "phase_offset_deg"]].values
y = units["accepted"].values
# 750 of 3,000 to test, the same share of accepted sensors in both parts (stratify).
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y,
                                          random_state=9101)
scaler = StandardScaler().fit(X_tr)          # fitted on the training sensors only
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)

drift = make_channel_drift()
D = drift[["drift_a_mv", "drift_b_mv"]].values
d = drift["correctable"].values

# Three parts of the digits: train, validation (to compare designs), test.
Xd_tr, yd_tr, Xd_va, yd_va, Xd_te, yd_te = load_digit_images()
print(f"sensors: {len(Z_tr)} to train, {len(Z_te)} to test; drift: {len(D)}; "
      f"digits: {len(Xd_tr)}, {len(Xd_va)}, {len(Xd_te)}")

I0000 00:00:1795778557.094190  578095 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1795778558.067286  578095 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


sensors: 2250 to train, 750 to test; drift: 800; digits: 1077, 360, 360


## 1. One unit by hand: a weighted sum, then the sigmoid

A unit computes $z = w \cdot x + b$ and passes it through the sigmoid. With $w = (1, 2)$ and $b = -1$, three points: one on each side of the line, one on it.

*From:* slides *The unit and its line*; *One unit, by hand*; handout section 2.1.

### Your prediction, before you run the next cell

Which of $(1, 1)$, $(0, 0)$ and $(1, 0)$ gets exactly 0.5, and why that one?

In [2]:
w, b = np.array([1.0, 2.0]), -1.0
for point in ([1, 1], [0, 0], [1, 0]):
    z = np.array(point) @ w + b              # the weighted sum: which side, how far
    y_hat = sigmoid(z)                       # squashed into (0, 1); the side does not change
    # Above 0.5 class 1, below class 0, exactly 0.5 on the line.
    label = "class 1" if y_hat > 0.5 else "class 0" if y_hat < 0.5 else "undecided"
    print(f"x = {point}:  z = {z:+.0f}   y-hat = {y_hat:.4f}   {label}")

x = [1, 1]:  z = +2   y-hat = 0.8808   class 1
x = [0, 0]:  z = -1   y-hat = 0.2689   class 0
x = [1, 0]:  z = +0   y-hat = 0.5000   undecided


**What to look at.** $(1, 1)$ gives $z = 2$ and $\hat{y} = 0.8808$; $(0, 0)$ gives $z = -1$ and 0.2689; $(1, 0)$ gives $z = 0$, so $\hat{y} = 0.5$: it lies on the line $x_1 + 2x_2 - 1 = 0$. The sigmoid never changes the side, only how sure.

**Try changing.** the point $(2, -0.5)$: work out $z$ on paper first, then add it to the list.

## 2. Why the pass region is a circle of radius 1.25

The rule divides each offset by its tolerance. Both tolerances are 1.25 times their axis's production spread, so after standardising the rule reads $u^2 + v^2 < 1.25^2$. The scaler uses the training sensors' own spreads, so the circle is very nearly, not exactly, the true boundary.

*From:* slides *Meridian's acceptance test*; *2,250 sensors, raw and standardised*; handout section 2.2.

### Your prediction, before you run the next cell

The rule ends in "< 1". Is the radius in standardised units 1, or something else?

In [3]:
# What the scaler learned in fit: the training sensors' mean and spread per axis.
print("means:  ", scaler.mean_.round(3), "  spreads:", scaler.scale_.round(3))
# Each tolerance divided by the spread the scaler actually uses: the semi-axes of
# the true boundary, drawn in the standardised coordinates.
semi_axes = np.array([0.50, 3.75]) / scaler.scale_
print("semi-axes of the true boundary, standardised:", semi_axes.round(2))
print("with the production spreads exactly:", np.array([0.50 / 0.40, 3.75 / 3.00]))

means:   [ 0.007 -0.031]   spreads: [0.39  3.057]
semi-axes of the true boundary, standardised: [1.28 1.23]
with the production spreads exactly: [1.25 1.25]


**What to look at.** The fitted spreads are close to 0.40 dB and 3.00 degrees but not equal to them, so the semi-axes come out at 1.28 and 1.23 instead of 1.25 and 1.25: very nearly a circle of radius 1.25. The radius is 1.25, not 1: the "1" in the rule counts tolerances, and each tolerance is 1.25 spreads.

**Try changing.** fit the scaler on all 3,000 sensors instead of the 2,250 and see how close to 1.25 the semi-axes come (and why that would be a leak in a real model).

## 3. What one line is worth on a disc

Logistic regression is one unit. Fit it, find where its line lies, and compare it with always answering "accepted".

*From:* slides *What one line is worth*; *No line encloses a region*; handout section 2.3.

### Your prediction, before you run the next cell

Above or below the baseline of 0.5480, and by how much? Where will the line be?

In [4]:
line = LogisticRegression().fit(Z_tr, y_tr)
(w_u, w_v), b0 = line.coef_[0], line.intercept_[0]   # coef_ has one row per class boundary
print(f"weights {w_u:.4f}, {w_v:.4f}, intercept {b0:.4f}")
# The line is where the weighted sum is zero: solve for v.
# w_u u + w_v v + b0 = 0  gives  v = -b0 / w_v - (w_u / w_v) u.
print(f"the line: v = {-b0 / w_v:.2f} {-w_u / w_v:+.2f} u")
print(f"logistic regression, test: {line.score(Z_te, y_te):.4f}")
# The baseline: always the most common class of the training sensors.
print(f"always 'accepted', test:   {(y_te == 1).mean():.4f}")

weights 0.0096, 0.0701, intercept 0.1892
the line: v = -2.70 -0.14 u
logistic regression, test: 0.5507
always 'accepted', test:   0.5480


**What to look at.** The weights are almost zero, and the line sits near $v = -2.7$, almost horizontal and far below the circle: the model says "accepted" to nearly everyone. 0.5507 against 0.5480 is two sensors of 750. The fit did not fail: by symmetry, whatever a line gains by tilting one way it loses on the other.

**Try changing.** fit on the raw offsets `X_tr` without the scaler and compare the score.

## 4. Two ReLU units written by hand, and the same network without the ReLU

The drift network of the slides: $h_1 = \max(0, a + b - 1)$, $h_2 = \max(0, -a - b - 1)$, $\hat{y} = \sigma(10h_1 + 10h_2 - 1)$. Then take the ReLU away and keep every weight.

*From:* slides *Two units, written down rather than trained*; *Without the ReLU, two layers are one*; handout section 3.2.

### Your prediction, before you run the next cell

Without the ReLU, what does the network say to the sensor $(1, 1)$, and to $(1, -1)$?

In [5]:
W1 = np.array([[1.0, -1.0],      # column 1: unit 1 sees a + b
               [1.0, -1.0]])     # column 2: unit 2 sees -a - b
b1 = np.array([-1.0, -1.0])
W2, b2 = np.array([10.0, 10.0]), -1.0

# D @ W1: the 800 x 2 drifts times the 2 x 2 weights, one row of (z1, z2) per sensor.
with_relu = sigmoid(relu(D @ W1 + b1) @ W2 + b2)     # the forward pass, two lines
# (with_relu > 0.5) is the predicted class; == d marks the right ones; the mean is the accuracy.
print(f"with the ReLU, all 800:    {((with_relu > 0.5) == d).mean():.4f}")

# Without an activation each hidden unit passes its weighted sum on unchanged.
z_lin = (D @ W1 + b1) @ W2 + b2
print("without it, z for the first five sensors:", z_lin[:5])
print(f"without it, all 800:       {((sigmoid(z_lin) > 0.5) == d).mean():.4f}")
# The algebra: one weight vector and one bias, W1 W2 and b1 W2 + b2.
print("collapsed weights W1 W2 =", W1 @ W2, "  collapsed bias =", b1 @ W2 + b2)

with the ReLU, all 800:    0.9938
without it, z for the first five sensors: [-21. -21. -21. -21. -21.]
without it, all 800:       0.5000
collapsed weights W1 W2 = [0. 0.]   collapsed bias = -21.0


**What to look at.** With the ReLU the network scores 0.9938. Without it every sensor gets $z = -21$: $a$ and $b$ cancel, the collapsed weights are $(0, 0)$ and the bias is $-21$, so the network says "not correctable" to all 800 and scores 0.5000. A weighted sum of weighted sums is one weighted sum; the bend is what matters.

**Try changing.** change the output weights to 10 and 5: what does the network without the ReLU compute now, and is it still a single line?

## 5. The hidden layer moved the data: one line is now enough

Compute the two hidden values for every sensor, then give them, and nothing else, to logistic regression.

*From:* slides *What the hidden layer actually did*; *The hidden layer classified nothing*; handout section 3.2.

### Your prediction, before you run the next cell

On the raw drifts a line scored 0.5000. On $(h_1, h_2)$, about how much?

In [6]:
H = relu(D @ W1 + b1)                        # the new coordinates (h1, h2)
for cloud in ([1, 1], [-1, -1], [1, -1], [-1, 1]):
    print(f"centre {cloud}: (h1, h2) =", relu(np.array(cloud) @ W1 + b1))
print(f"\none line on the raw drifts: {LogisticRegression().fit(D, d).score(D, d):.4f}")
print(f"one line on (h1, h2):       {LogisticRegression().fit(H, d).score(H, d):.4f}")

centre [1, 1]: (h1, h2) = [1. 0.]
centre [-1, -1]: (h1, h2) = [0. 1.]
centre [1, -1]: (h1, h2) = [0. 0.]
centre [-1, 1]: (h1, h2) = [0. 0.]

one line on the raw drifts: 0.5000
one line on (h1, h2):       0.9738


**What to look at.** The correctable centres go to $(1, 0)$ and $(0, 1)$, the others to $(0, 0)$. On the raw drifts one line is worth nothing, 0.5000; on the new coordinates one line scores 0.9738. The hidden layer classified nothing; it changed the coordinates, and here we wrote them by hand.

**Try changing.** use only $h_1$ as the single input: which clouds can one threshold still separate?

## 6. Training finds the weights, sometimes

`MLPClassifier` with two hidden units, the size that can solve the drift problem, from eight random starts. `loss_` is the cross-entropy where each run stopped.

*From:* slides *Letting training find the weights*; *Two units can; training often misses them*; handout section 3.4.

### Your prediction, before you run the next cell

Two units can solve it. Will all eight runs reach 0.99, and will they stop at the same loss?

In [7]:
for seed in range(8):
    net = MLPClassifier(hidden_layer_sizes=(2,), max_iter=3000, random_state=seed).fit(D, d)
    # Scored on the 800 it trained on: the question is whether training finds the lines.
    print(f"random_state {seed}: accuracy {net.score(D, d):.4f}   final loss {net.loss_:.4f}")

random_state 0: accuracy 0.7425   final loss 0.3715
random_state 1: accuracy 0.7375   final loss 0.5019


random_state 2: accuracy 0.7438   final loss 0.3699
random_state 3: accuracy 0.5000   final loss 0.6985


random_state 4: accuracy 0.7375   final loss 0.3686


random_state 5: accuracy 0.9938   final loss 0.0587


random_state 6: accuracy 0.9938   final loss 0.0602
random_state 7: accuracy 0.7262   final loss 0.5105


**What to look at.** Two runs of eight reach 0.9938 and stop at a loss near 0.06; five stop near 0.74 at a loss six times larger, having walled off three clouds of four, and one is stuck at 0.5000. The same network, different starting weights, different valleys: a network's cost has many, and each of those runs stopped in a local minimum.

**Try changing.** `hidden_layer_sizes=(8,)`: how many of the eight runs succeed now?

## 7. Each hidden unit is a line; the corners sit on the lines

Eight hidden units on the acceptance test, run 0. For each unit: how far its line is from the centre, its output weight, and whether its line passes through the region the network accepts.

*From:* slides *The boundary at 1, 2, 4 and 8 units*; *Each hidden unit is a line*; handout section 3.4.

### Your prediction, before you run the next cell

Eight units: will all eight lines cut through the accepted region?

In [8]:
net = MLPClassifier(hidden_layer_sizes=(8,), max_iter=3000, random_state=0).fit(Z_tr, y_tr)
print(f"test accuracy {net.score(Z_te, y_te):.4f}")
W, bias = net.coefs_[0], net.intercepts_[0]       # column j: hidden unit j's line
out_w = net.coefs_[1].ravel()                     # how the output weighs each unit
# A fine grid over the plane: 301 x 301 points, every (u, v) pair.
grid = np.linspace(-3, 3, 301)
G = np.array([(u, v) for u in grid for v in grid])
inside = G[net.predict(G) == 1]                   # keep the points the network accepts
crossing = 0
for j in range(8):
    side = inside @ W[:, j] + bias[j] > 0         # which side of unit j's line
    cuts = side.any() and not side.all()          # points on both sides: it crosses
    crossing += cuts
    # Distance of the line w.x + b = 0 from the origin: |b| divided by the length of w.
    print(f"unit {j}: {abs(bias[j]) / np.linalg.norm(W[:, j]):.2f} from the centre, "
          f"output weight {out_w[j]:+.2f}, {'crosses' if cuts else 'misses'}")
print(f"{crossing} lines cross, two corners each: {2 * crossing} corners")

test accuracy 0.9493
unit 0: 0.01 from the centre, output weight -2.49, crosses
unit 1: 1.84 from the centre, output weight +0.96, misses
unit 2: 0.06 from the centre, output weight -2.14, crosses
unit 3: 2.83 from the centre, output weight +1.72, misses
unit 4: 2.68 from the centre, output weight +1.02, misses
unit 5: 0.05 from the centre, output weight -1.55, crosses
unit 6: 0.19 from the centre, output weight -2.00, crosses
unit 7: 4.38 from the centre, output weight +0.68, misses
4 lines cross, two corners each: 8 corners


**What to look at.** Four lines pass close to the centre with negative output weights and cross the region; the other four lie far out. Between two lines the same units are on, so the boundary is straight; crossing a line switches a unit, and the boundary turns a corner: 4 lines, twice each, 8 corners. Not one side per unit.

**Try changing.** `random_state=1`: count again. Is it still four crossing lines?

## 8. Mini-batches and epochs on the sensors

`MLPClassifier`'s default mini-batch is `min(200, m)`. Shuffle the 2,250 training sensors and cut them into blocks, as one epoch does.

*From:* slides *How a network learns: the loop*; *Mini-batches and epochs*; handout section 4.1.

### Your prediction, before you run the next cell

How many mini-batches in one epoch, and how big is the last one?

In [9]:
print("MLPClassifier's batch_size:", MLPClassifier().get_params()["batch_size"])
batch = min(200, len(Z_tr))
order = np.random.default_rng(0).permutation(len(Z_tr))     # shuffled once per epoch
# Consecutive slices of 200 from the shuffled order; the last slice keeps what is left.
blocks = [order[i:i + batch] for i in range(0, len(order), batch)]
print(f"{len(Z_tr)} / {batch} = {len(Z_tr) / batch}: {len(blocks)} mini-batches")
print("sizes:", [len(blk) for blk in blocks])
# Within one epoch every sensor is used exactly once.
# Glue the blocks back, sort, and compare with 0, 1, ..., 2,249: each index exactly once.
print("every sensor once:", np.array_equal(np.sort(np.concatenate(blocks)), np.arange(len(Z_tr))))

MLPClassifier's batch_size: auto
2250 / 200 = 11.25: 12 mini-batches
sizes: [200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 50]
every sensor once: True


**What to look at.** `'auto'` means $\min(200, m) = 200$; $2{,}250 / 200 = 11.25$, so 12 mini-batches, 11 of 200 and one of 50, and every sensor appears exactly once per epoch. An epoch is 12 updates, not one.

**Try changing.** Keras's default of 32 on the 1,077 training digits: how many mini-batches per epoch?

## 9. One gradient step on one unit, and what "times its input" means

The unit of atom 1 at $(1, 1)$ with true label 0, then at $(1, 0)$. Each weight's blame is $(\hat{y} - y)$ times its input; check it against nudging the weight and measuring the loss.

*From:* slides *The loss and the gradient step*; *One step on one unit*; *Reading the step*; handout section 4.3.

### Your prediction, before you run the next cell

At $(1, 0)$, which weight does not move at all?

In [10]:
def loss(w_, b_, x_, y_):
    p = sigmoid(x_ @ w_ + b_)
    return -(y_ * np.log(p) + (1 - y_) * np.log(1 - p))     # cross-entropy, lesson 4

alpha = 0.5
for x in (np.array([1.0, 1.0]), np.array([1.0, 0.0])):
    y_hat = sigmoid(x @ w + b)
    blame = (y_hat - 0) * np.append(x, 1)                   # w1, w2, then the bias (input 1)
    # The chain rule's answer, against a numerical derivative of the loss for w1.
    h = 1e-6
    # Central difference: loss a hair above w1 minus a hair below, over the distance 2h.
    numeric = (loss(w + [h, 0], b, x, 0) - loss(w - [h, 0], b, x, 0)) / (2 * h)
    w_new, b_new = w - alpha * blame[:2], b - alpha * blame[2]
    print(f"x = {x}: y-hat {y_hat:.4f}, blames {blame.round(4)}, "
          f"dL/dw1 by nudging {numeric:.4f}")
    print(f"   new weights {w_new}, bias {b_new:.4f}, new y-hat {sigmoid(x @ w_new + b_new):.4f}")

x = [1. 1.]: y-hat 0.8808, blames [0.8808 0.8808 0.8808], dL/dw1 by nudging 0.8808
   new weights [0.5596 1.5596], bias -1.4404, new y-hat 0.6635
x = [1. 0.]: y-hat 0.5000, blames [0.5 0.  0.5], dL/dw1 by nudging 0.5000
   new weights [0.75 2.  ], bias -1.2500, new y-hat 0.3775


**What to look at.** At $(1, 1)$ every input is 1, so all three blames are 0.8808 and the factor is invisible; the step takes $\hat{y}$ from 0.881 to 0.663. At $(1, 0)$ the blames are 0.5, 0 and 0.5: $w_2$ stays at 2, because its input was 0 and it took no part. The nudged derivative agrees with the formula.

**Try changing.** set the true label to 1 at $(1, 1)$: which way do the weights move now?

## 10. Backpropagation by hand, and what it buys

The drift network on the sensor $(1.5, -0.1)$, $y = 0$. The backward sweep gives every weight's blame at once; the naive way nudges one weight at a time and reruns the forward pass for each.

*From:* slides *Backpropagation by hand*; *The blame, sent back with every number*; *One step with α = 0.01*; *What backpropagation buys*; handout section 4.4.

### Your prediction, before you run the next cell

How many forward passes does the naive way need for this network's 9 weights?

In [11]:
s = np.array([1.5, -0.1])
params = [W1.copy(), b1.copy(), W2.copy(), np.array(b2)]
passes = 0


def forward_loss(W1_, b1_, W2_, b2_):
    global passes
    passes += 1                                    # count every forward pass
    return -np.log(1 - sigmoid(relu(s @ W1_ + b1_) @ W2_ + b2_))


# Backward sweep: one forward pass, then the chain rule from the output back.
z1 = s @ W1 + b1
y_hat = sigmoid(relu(z1) @ W2 + b2)
d_out = y_hat - 0                                  # 1. at the output
d_hidden = d_out * W2 * (z1 > 0)                   # 2-3. back along the weights, times the ReLU's slope
# 4. np.outer: row i is input i, column j is unit j, so entry (i, j) = input i x blame of unit j.
blame_W1 = np.outer(s, d_hidden)
print(f"y-hat {y_hat:.4f}; blame at each hidden unit {d_hidden.round(4)}")
print("blame of W1 by backpropagation:\n", blame_W1.round(4))

# The naive way: nudge each of the 9 weights, rerun the forward pass, twice each.
passes, h, numeric = 0, 1e-6, []
for k, p in enumerate(params):
    for idx in np.ndindex(p.shape):               # every position in this weight array
        # Two copies of all the weights, this one nudged up in one and down in the other.
        up = [q.copy() for q in params]; up[k][idx] += h
        down = [q.copy() for q in params]; down[k][idx] -= h
        numeric.append((forward_loss(*up) - forward_loss(*down)) / (2 * h))
# The first four derivatives belong to W1, in row order: reshape them to 2 x 2.
print("blame of W1 by nudging:\n", np.array(numeric[:4]).reshape(2, 2).round(4))
print(f"forward passes the naive way: {passes}, for {len(numeric)} weights")

y-hat 0.9526; blame at each hidden unit [9.5257 0.    ]
blame of W1 by backpropagation:
 [[14.2886  0.    ]
 [-0.9526 -0.    ]]
blame of W1 by nudging:
 [[14.2886  0.    ]
 [-0.9526  0.    ]]
forward passes the naive way: 18, for 9 weights


**What to look at.** Unit 1 receives $0.953 \times 10 = 9.53$; unit 2, off, receives 0. The weights into unit 1 get $9.53 \times 1.5 = 14.29$ and $9.53 \times (-0.1) = -0.95$; nudging gives the same numbers, at the price of 18 forward passes for 9 weights. The backward sweep costs about one.

**Try changing.** take one step with $\alpha = 0.01$ (every weight minus 0.01 times its blame) and check that $\hat{y}$ drops below 0.5, to 0.471.

## 11. Softmax by hand, and with two classes

Softmax exponentiates every score and divides by the total. Check it against Keras's own softmax, then show that only differences of scores matter.

*From:* slides *Softmax*; *Softmax in words, with three scores*; *Two classes: softmax is the sigmoid*; *Why the difference of the two scores*; handout section 5.3.

### Your prediction, before you run the next cell

Add 5 to every score of $(2, 1, 0)$: do the probabilities change?

In [12]:
def softmax(z):
    e = np.exp(z)
    return e / e.sum()                        # one shared denominator: they add to 1

z = np.array([2.0, 1.0, 0.0])
print("by hand:     ", softmax(z).round(4), " sum", softmax(z).sum())
print("Keras:       ", keras.ops.softmax(z).numpy().round(4))   # .numpy(): tensor to array
print("scores + 5:  ", softmax(z + 5).round(4))
# Two classes: class 1's probability is the sigmoid of the difference of the scores.
for pair in ([0.0, 1.0], [5.0, 6.0]):
    print(f"scores {pair}: softmax gives class 1 {softmax(np.array(pair))[1]:.4f}, "
          f"sigma(z1 - z0) = {sigmoid(pair[1] - pair[0]):.4f}")

by hand:      [0.6652 0.2447 0.09  ]  sum 1.0
Keras:        [0.6652 0.2447 0.09  ]
scores + 5:   [0.6652 0.2447 0.09  ]
scores [0.0, 1.0]: softmax gives class 1 0.7311, sigma(z1 - z0) = 0.7311
scores [5.0, 6.0]: softmax gives class 1 0.7311, sigma(z1 - z0) = 0.7311


**What to look at.** 0.665, 0.245 and 0.090, the same from Keras, and unchanged when 5 is added to every score: only differences count. With two classes both $(0, 1)$ and $(5, 6)$ give 0.731, which is $\sigma(1)$: two output units are one unit, logistic regression.

**Try changing.** give the scores $(20, 0, 0)$: how close to 1 does the first probability get, and is it ever exactly 1?

## 12. Counting the parameters, and the baseline of the digits

Build the digit networks in Keras and read their parameter counts; then score the baseline, always answering the most common training digit.

*From:* slides *The digit network, drawn*; *A network in Keras: 2,410 parameters*; *Is the hidden layer worth it here?*; handout sections 5.2 and 5.5.

### Your prediction, before you run the next cell

64 inputs, 32 hidden units, 10 outputs: how many weights and biases in all?

In [13]:
for hidden in ([], [32], [64]):
    # Input, then one Dense layer per hidden width (none, one of 32, one of 64), then 10 outputs.
    model = keras.Sequential([keras.Input(shape=(64,))]
                             + [layers.Dense(u, activation="relu") for u in hidden]
                             + [layers.Dense(10, activation="softmax")])
    print(f"hidden {str(hidden):>5}: {model.count_params():,} parameters")
print("by hand, one layer of 32:", 64 * 32 + 32 + 32 * 10 + 10)

counts = np.bincount(yd_tr)                     # entry k: how many training images show digit k
print("\ntraining images per digit:", counts)
common = counts.argmax()                        # ties go to the first: 1, 3, 5, 6 all have 109
print(f"baseline, always {common}: validation {(yd_va == common).mean():.4f}")
print("validation images per digit (the confusion matrix's row sums):", np.bincount(yd_va))

hidden    []: 650 parameters
hidden  [32]: 2,410 parameters
hidden  [64]: 4,810 parameters
by hand, one layer of 32: 2410

training images per digit: [107 109 106 109 109 109 109 107 104 108]
baseline, always 1: validation 0.1028
validation images per digit (the confusion matrix's row sums): [35 37 36 37 36 36 36 36 35 36]


**What to look at.** 650, 2,410 and 4,810 parameters; $64 \times 32 + 32 + 32 \times 10 + 10 = 2{,}410$. The baseline answers 1 and scores $37 / 360 = 0.1028$: it says whether a model learned anything. The rows of the confusion matrix add up to these validation counts, 35 to 37.

**Try changing.** a second hidden layer of 32: predict the new count before running it.

## 13. Learning rates a factor of ten apart

Plain SGD, two hidden layers of 64, the same network and start at four rates, ten epochs each: a short version of notebook 03's experiment.

*From:* slides *Two layers of 64 on the digits, at three learning rates*; *A rate too small looks like a network too small*; handout section 6.1.

### Your prediction, before you run the next cell

Which of 0.001, 0.01, 0.1 and 1 will be best after ten epochs?

In [14]:
def digit_model(optimizer, seed=0):
    keras.utils.set_random_seed(seed)            # the same starting weights every time
    model = keras.Sequential([keras.Input(shape=(64,)),
                              layers.Dense(64, activation="relu"),
                              layers.Dense(64, activation="relu"),
                              layers.Dense(10, activation="softmax")])
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


for rate in (0.001, 0.01, 0.1, 1.0):             # multiplications, not additions
    model = digit_model(keras.optimizers.SGD(rate))
    history = model.fit(Xd_tr, yd_tr, validation_data=(Xd_va, yd_va),
                        epochs=10, batch_size=32, verbose=0).history
    print(f"alpha {rate:<5}: training loss {history['loss'][-1]:.4f}, "
          f"validation {history['val_accuracy'][-1]:.4f}")

alpha 0.001: training loss 2.2715, validation 0.1222


alpha 0.01 : training loss 1.8107, validation 0.6972


alpha 0.1  : training loss 0.1500, validation 0.9472


alpha 1.0  : training loss 2.3383, validation 0.1000


**What to look at.** After ten epochs 0.001 is still at 0.1222 and 0.01 at 0.6972, low on both sets, which looks exactly like a network too small; the same network reaches 0.9472 at 0.1, and at 1.0 it is too large and stuck at 0.1000. A factor of ten between tries covers four orders of magnitude in four runs.

**Try changing.** replace `keras.optimizers.SGD(rate)` with `keras.optimizers.Adam(0.001)`: why does Adam manage at a rate SGD finds too small?

## 14. A rising loss with a flat accuracy

An image's loss is $-\log$ of the probability given to its true class. Take three images, the network wrong on one of them, and make it more and more sure of its answers.

*From:* slides *Two layers of 512 on 300 digits*; *Reading the two curves*; handout section 6.2.

### Your prediction, before you run the next cell

If the network grows surer of its wrong answer, does the loss go up or down? Does the accuracy change?

In [15]:
true = np.array([0, 1, 2])
for sure in (0.6, 0.9, 0.99):
    rest = (1 - sure) / 2
    # Images 0 and 1 right with probability `sure`; image 2 wrong: `sure` goes to class 0.
    # One row per image, one column per class; each row adds up to 1.
    p = np.array([[sure, rest, rest], [rest, sure, rest], [sure, rest, rest]])
    accuracy = (p.argmax(axis=1) == true).mean()
    # p[np.arange(3), true] picks, row by row, the probability of that image's true class.
    losses = -np.log(p[np.arange(3), true])
    print(f"sure {sure}: accuracy {accuracy:.3f}, losses {losses.round(2)}, mean {losses.mean():.3f}")
print(f"\n-log 0.10 = {-np.log(0.10):.2f},  -log 0.01 = {-np.log(0.01):.2f}")

sure 0.6: accuracy 0.667, losses [0.51 0.51 1.61], mean 0.877
sure 0.9: accuracy 0.667, losses [0.11 0.11 3.  ], mean 1.069
sure 0.99: accuracy 0.667, losses [0.01 0.01 5.3 ], mean 1.773

-log 0.10 = 2.30,  -log 0.01 = 4.61


**What to look at.** The accuracy stays at 0.667 throughout. The right answers' losses fall towards 0, but the wrong one's rises without limit, and it pulls the mean up: a rising validation loss with a flat accuracy means the network is surer of the answers it gets wrong, not less sure.

**Try changing.** make image 2 right instead and watch the mean loss as `sure` grows.

## 15. Dropout, looked at directly

Keras's `Dropout(0.4)` applied to ten activations of 1, in training mode and in prediction mode.

*From:* slides *Two remedies for overfitting*; *Dropout, drawn*; handout section 6.3.

### Your prediction, before you run the next cell

In training, what values come out besides 0? In prediction?

In [16]:
drop = layers.Dropout(0.4, seed=0)
ones = np.ones((3, 10), dtype="float32")         # three examples, ten activations each
print("training, three examples:\n", drop(ones, training=True).numpy().round(3))
print("prediction:\n", drop(ones, training=False).numpy())
print("the scale for the survivors, 1 / 0.6 =", round(1 / 0.6, 3))

training, three examples:
 [[0.    1.667 1.667 1.667 0.    1.667 1.667 0.    1.667 1.667]
 [0.    0.    1.667 0.    0.    1.667 1.667 0.    0.    0.   ]
 [0.    1.667 1.667 0.    1.667 0.    0.    0.    0.    1.667]]
prediction:
 [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]]
the scale for the survivors, 1 / 0.6 = 1.667


**What to look at.** In training each value is dropped with probability 0.4 (here 15 of the 30, by chance), a different set for each example, and the survivors are 1.667, so the total stays the same on average; in prediction nothing is dropped. Nothing is removed for good.

**Try changing.** `Dropout(0.9)`: what is the survivors' value now, and why would training struggle?

## 16. The vanishing gradient, measured

Two untrained networks of six hidden layers of 32, identical but for the activation. `tf.GradientTape` gives the blame of every weight for one mini-batch; compare its size in the first layer and the last.

*From:* slides *The sigmoid's slope is at most 1/4*; *Six sigmoid layers, plain SGD*; *The vanishing gradient*; handout section 6.4.

### Your prediction, before you run the next cell

With sigmoids, how much smaller is the first layer's blame than the last layer's?

In [17]:
print(f"the sigmoid's largest slope: {sigmoid(0) * (1 - sigmoid(0))}; six layers: 0.25**6 = {0.25 ** 6:.6f}")
for activation in ("sigmoid", "relu"):
    keras.utils.set_random_seed(0)
    model = keras.Sequential([keras.Input(shape=(64,))]
                             + [layers.Dense(32, activation=activation) for _ in range(6)]
                             + [layers.Dense(10, activation="softmax")])
    with tf.GradientTape() as tape:              # record one forward pass on 32 images
        p = model(Xd_tr[:32])
        # The ten-class cross-entropy written out: minus log of the true digit's probability.
        onehot = np.eye(10)[yd_tr[:32]]          # row i: a 1 in the column of image i's digit
        loss = -tf.reduce_sum(onehot * tf.math.log(p), axis=1)
    grads = tape.gradient(loss, model.trainable_weights)
    # trainable_weights alternate kernel, bias per layer: [0] is the first hidden kernel,
    # [-4] the last hidden kernel ([-2] and [-1] belong to the output layer).
    first, last = grads[0], grads[-4]
    size = lambda g: float(np.abs(g.numpy()).mean())   # typical size: mean absolute blame
    print(f"{activation:7s}: first layer {size(first):.2e}, last hidden {size(last):.2e}, "
          f"ratio {size(first) / size(last):.4f}")

the sigmoid's largest slope: 0.25; six layers: 0.25**6 = 0.000244
sigmoid: first layer 7.40e-05, last hidden 1.22e-01, ratio 0.0006
relu   : first layer 2.40e-02, last hidden 4.17e-02, ratio 0.5744


**What to look at.** With sigmoids the first layer's blame is 0.0006 of the last hidden layer's: each layer multiplies it by a slope of at most 1/4. With the ReLU the ratio is 0.57, the same order. The first layers of the sigmoid network barely move, which is why it stayed at chance.

**Try changing.** six layers of 128 instead of 32: does the ratio change much?

## 17. Shuffled pixels: a linear model does not notice either

One fixed reordering of the 64 pixels, applied to every image. Logistic regression gives each pixel its own weight, so it should learn the same weights in the shuffled order.

*From:* slides *Shuffle the pixels the same way in every image, then train again*; *Nothing notices*; handout section 6.5.

### Your prediction, before you run the next cell

Same validation accuracy, lower, or higher?

In [18]:
permutation = np.random.default_rng(10).permutation(64)
plain = LogisticRegression(max_iter=5000).fit(Xd_tr, yd_tr)
shuffled = LogisticRegression(max_iter=5000).fit(Xd_tr[:, permutation], yd_tr)
print(f"as written:      {plain.score(Xd_va, yd_va):.4f}")
print(f"pixels shuffled: {shuffled.score(Xd_va[:, permutation], yd_va):.4f}")
# The weights are the same, in the shuffled order.
# plain.coef_[:, permutation] reorders the original weights the way the pixels were
# reordered; if the model learned the same thing, it matches shuffled.coef_.
print("largest weight difference:",
      float(np.abs(plain.coef_[:, permutation] - shuffled.coef_).max().round(4)))
# The predictable slip: train on the originals, shuffle only the validation images.
print(f"trained as written, scored shuffled: {plain.score(Xd_va[:, permutation], yd_va):.4f}")

as written:      0.9556
pixels shuffled: 0.9556
largest weight difference: 0.0001
trained as written, scored shuffled: 0.0639


**What to look at.** The two scores agree, 0.9556, and the weights are the same ones in the shuffled order: nothing in a model that gives every pixel its own weight knows which pixel is next to which. Train on the originals and shuffle only the validation images, and the score collapses to 0.0639: the shuffle must be the same everywhere.

**Try changing.** a different permutation seed: does anything change?